# Project 3 - FMNN25

### Task 1 - Design an optimization problem class in Python. Its constructor should take an objective function as input, optionally also its gradient can be provided.

In [18]:
from collections.abc import Callable

import numpy as np


class OptimizationProblem:
    """Store an objective function and, optionally, its gradient."""

    def __init__(
        self,
        objective_function: Callable[[np.ndarray], float],
        gradient_function: Callable[[np.ndarray], np.ndarray] | None = None,
    ) -> None:
        
        self.objective_function = objective_function

        # The gradient is optional since some methods can approximate it numerically
        self.gradient_function = gradient_function

    def objective(self, x: np.ndarray) -> float:
        """Evaluate the objective function at the point x."""
        x = np.asarray(x, dtype=float) #convert to numby array of floats

        return float(self.objective_function(x))

    def gradient(self, x: np.ndarray) -> np.ndarray:
        """Evaluate the gradient at the point x, by central differences if none was provided."""
        x = np.asarray(x, dtype=float)

        if self.gradient_function is None:
            h = 1e-6
            return np.array([
                (self.objective(x + h * e) - self.objective(x - h * e)) / (2 * h)
                for e in np.eye(len(x))
            ])

        return np.asarray(self.gradient_function(x), dtype=float)

    def __call__(self, x: np.ndarray) -> float:
        """Allow the problem object to be called like a function."""
        return self.objective(x)

test/example use 

In [19]:
def objective(x: np.ndarray) -> float:
    return (x[0] - 1) ** 2 + (x[1] - 2) ** 2


def gradient(x: np.ndarray) -> np.ndarray:
    return np.array([
        2 * (x[0] - 1),
        2 * (x[1] - 2),
    ])


problem = OptimizationProblem(objective, gradient)

x = np.array([0.0, 0.0])

print(problem.objective(x))
print(problem.gradient(x))
print(problem(x))

5.0
[-2. -4.]
5.0


### Task 2 - Design a general optimization method class. It should be generic for special kinds of Quasi-Newton methods. Derive from this method class classes for special methods in the next tasks.

`OptimizationMethod` owns the iteration loop: stopping criterion, line search, step and the history of iterates. A subclass only defines `search_direction` and, for Quasi-Newton methods, `update` to refresh its Hessian approximation after each step.

In [20]:
class OptimizationMethod:
    """Generic iteration x_{k+1} = x_k + alpha_k * s_k, where subclasses supply the search direction s_k."""

    def __init__(
        self,
        problem: OptimizationProblem,
        line_search: Callable[[OptimizationProblem, np.ndarray, np.ndarray], float] | None = None,
        tol: float = 1e-6,
        max_iter: int = 1000,
    ) -> None:
        self.problem = problem
        self.line_search = line_search  # None means the full step alpha = 1
        self.tol = tol
        self.max_iter = max_iter
        self.history = []

    def search_direction(self, x: np.ndarray, g: np.ndarray) -> np.ndarray:
        """Return the search direction at x, where g is the gradient at x."""
        raise NotImplementedError

    def update(self, delta: np.ndarray, gamma: np.ndarray) -> None:
        """Hook for Quasi-Newton methods to update their Hessian approximation after a step.

        Here delta = x_new - x and gamma = g_new - g.
        """

    def solve(self, x0: np.ndarray) -> np.ndarray:
        """Iterate from x0 until the norm of the gradient drops below tol and return the final point."""
        x = np.asarray(x0, dtype=float)
        g = self.problem.gradient(x)
        self.history = [x]

        for _ in range(self.max_iter):
            if np.linalg.norm(g) < self.tol:
                break

            s = self.search_direction(x, g)
            alpha = 1.0 if self.line_search is None else self.line_search(self.problem, x, s)
            x_new = x + alpha * s
            g_new = self.problem.gradient(x_new)

            self.update(x_new - x, g_new - g)
            x, g = x_new, g_new
            self.history.append(x)

        return x

### Task 3 - Implement classical Newton's method for finding a minimum of the objective function. Approximate the Hessian by finite differences and a symmetrizing step: $G := \frac{1}{2}(\bar G + \bar G^T)$, if necessary.

`Newton` derives from `OptimizationMethod` and only defines the search direction $s = -G^{-1} g$. Without a line search it takes the full step $\alpha = 1$, which is the classical method.

In [21]:
class Newton(OptimizationMethod):
    """Newton's method with the Hessian approximated by finite differences of the gradient."""

    def __init__(
        self,
        problem: OptimizationProblem,
        line_search: Callable[[OptimizationProblem, np.ndarray, np.ndarray], float] | None = None,
        tol: float = 1e-6,
        max_iter: int = 1000,
        h: float = 1e-6,
    ) -> None:
        super().__init__(problem, line_search, tol, max_iter)
        self.h = h

    def hessian(self, x: np.ndarray) -> np.ndarray:
        """Approximate the Hessian at x by central differences and symmetrize it."""
        G = np.column_stack([
            (self.problem.gradient(x + self.h * e) - self.problem.gradient(x - self.h * e)) / (2 * self.h)
            for e in np.eye(len(x))
        ])

        return 0.5 * (G + G.T)

    def search_direction(self, x: np.ndarray, g: np.ndarray) -> np.ndarray:
        """Solve G s = -g for the Newton direction s."""
        return np.linalg.solve(self.hessian(x), -g)

Test on the quadratic problem from Task 1: the Hessian should be $2I$ and Newton's method should converge in a single step.

In [22]:
newton = Newton(problem)
x_min = newton.solve(np.array([0.0, 0.0]))

print(newton.hessian(x_min))
print(x_min, len(newton.history) - 1)

[[2. 0.]
 [0. 2.]]
[1. 2.] 1


### Task 4 - Provide Newton's method with exact line search method.

A line search is any function `(problem, x, s) -> alpha`, passed to the method through `line_search`. The exact one minimizes the one-dimensional function $\phi(\alpha) = f(x + \alpha s)$ with `scipy.optimize.minimize_scalar`.

In [23]:
from scipy.optimize import minimize_scalar


def exact_line_search(problem: OptimizationProblem, x: np.ndarray, s: np.ndarray) -> float:
    """Return the step length alpha that minimizes f(x + alpha * s)."""
    return minimize_scalar(lambda alpha: problem(x + alpha * s)).x

Test on $f(x) = (x_0 - 1)^4 + (x_1 - 2)^4$. The full Newton step only reduces the error by a factor $2/3$ per iteration, but the Newton direction points at the minimum, so the exact line search should reach it in one step.

In [24]:
quartic = OptimizationProblem(
    lambda x: (x[0] - 1) ** 4 + (x[1] - 2) ** 4,
    lambda x: np.array([4 * (x[0] - 1) ** 3, 4 * (x[1] - 2) ** 3]),
)
x0 = np.array([3.0, 0.0])

for name, line_search in [("full step", None), ("exact line search", exact_line_search)]:
    newton = Newton(quartic, line_search=line_search)
    x_min = newton.solve(x0)
    print(f"{name}: x = {x_min}, iterations = {len(newton.history) - 1}")

full step: x = [1.00456732 1.99543268], iterations = 15
exact line search: x = [1. 2.], iterations = 1
